# Objective 3 — Step 6: Cost-Sensitive XGBoost Ablation

Step 5 showed that the frozen German-developed Group-Aware Chi-Square Top-75% rule transferred almost losslessly to Taiwan, but caused a modest performance decrease on Australian Credit Approval.

Therefore, **feature selection will not yet be hard-coded as mandatory in the final hybrid framework**. This step evaluates the interaction between:

## Feature regimes
1. **All Features**
2. **Frozen Group-Aware Chi-Square Top-75%**

## Training-cost regimes
1. **Unweighted**: `scale_pos_weight = 1`
2. **Balanced**: `scale_pos_weight = N_negative / N_positive`
3. **Cost2**: `2 × N_negative / N_positive`
4. **Cost5**: `5 × N_negative / N_positive`

The class weight is calculated **only from each outer-training fold**.

## Validation
- Same 5 × 5 StratifiedGroupKFold outer folds used in Steps 3–5.
- No outer-test information is used for feature ranking or class weighting.
- Same fixed XGBoost structural hyperparameters as Steps 3–5.
- Default classification threshold remains 0.50.
- Probability calibration and threshold optimisation are deliberately deferred to later steps.

The experiment produces a controlled 2 × 4 ablation on each of the three datasets.


In [1]:
%pip install pandas numpy scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
from itertools import combinations
import json
import math
import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import chi2
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"D:\PHD\Research Paper writing\3rd Obj. paper")

STEP2_DIR = BASE_DIR / "results" / "preprocessing_protocol"
BASELINE_DIR = BASE_DIR / "results" / "baseline_models"
STEP4_DIR = BASE_DIR / "results" / "feature_selection_german_development"
STEP5_DIR = BASE_DIR / "results" / "feature_selection_frozen_replication"
DATA_DIR = BASE_DIR / "data" / "processed"

OUT_DIR = BASE_DIR / "results" / "cost_sensitive_xgb_ablation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DICTIONARY_FILE = STEP2_DIR / "preprocessing_data_dictionary.csv"
BASELINE_RESULTS_FILE = BASELINE_DIR / "baseline_fold_results_all.csv"
BASELINE_PREDICTIONS_FILE = BASELINE_DIR / "baseline_predictions_all.csv"
STEP4_RESULTS_FILE = STEP4_DIR / "german_feature_selection_fold_results.csv"
STEP5_RESULTS_FILE = STEP5_DIR / "frozen_replication_fold_results.csv"

DATASET_FILES = {
    "Australian Credit Approval":
        DATA_DIR / "australian_credit_approval_cleaned.csv",
    "German Credit":
        DATA_DIR / "german_credit_cleaned.csv",
    "Taiwan Credit Card Default":
        DATA_DIR / "taiwan_credit_card_default_cleaned.csv",
}

required = [
    DICTIONARY_FILE,
    BASELINE_RESULTS_FILE,
    BASELINE_PREDICTIONS_FILE,
    STEP4_RESULTS_FILE,
    STEP5_RESULTS_FILE,
    *DATASET_FILES.values(),
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Previous-step files are missing:\n" + "\n".join(missing)
    )

REPEAT_SEEDS = [42, 142, 242, 342, 442]
N_FOLDS = 5
FROZEN_FRACTION = 0.75

WEIGHT_STRATEGIES = {
    "Unweighted": 0.0,   # special case -> exactly 1.0
    "Balanced": 1.0,
    "Cost2": 2.0,
    "Cost5": 5.0,
}

FEATURE_REGIMES = [
    "AllFeatures",
    "FrozenChi2Top75",
]

print("Output folder:", OUT_DIR)


Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\cost_sensitive_xgb_ablation


## 1. Load datasets and feature roles

In [3]:

dictionary = pd.read_csv(DICTIONARY_FILE)
baseline_results = pd.read_csv(BASELINE_RESULTS_FILE)
baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_FILE)
step4_results = pd.read_csv(STEP4_RESULTS_FILE)
step5_results = pd.read_csv(STEP5_RESULTS_FILE)

datasets = {
    name: pd.read_csv(path)
    for name, path in DATASET_FILES.items()
}

def roles_for(dataset_name):
    d = dictionary[dictionary["dataset"] == dataset_name].copy()
    return {
        role: d.loc[d["role"] == role, "variable"].astype(str).tolist()
        for role in ["categorical", "ordinal", "numerical", "identifier"]
    }

roles = {name: roles_for(name) for name in datasets}

for dataset_name, df in datasets.items():
    r = roles[dataset_name]
    predictors = r["categorical"] + r["ordinal"] + r["numerical"]

    print(
        dataset_name,
        "| N =", len(df),
        "| predictors =", len(predictors),
        "| Top75 =", math.ceil(len(predictors) * FROZEN_FRACTION),
        "| adverse rate =", round(df["adverse_target"].mean(), 4),
    )


Australian Credit Approval | N = 690 | predictors = 14 | Top75 = 11 | adverse rate = 0.5551
German Credit | N = 1000 | predictors = 20 | Top75 = 15 | adverse rate = 0.3
Taiwan Credit Card Default | N = 30000 | predictors = 23 | Top75 = 18 | adverse rate = 0.2212


## 2. Fold-wise preprocessing

In [4]:

def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float32,
        )


def build_preprocessor(dataset_name, mode, selected_features=None):
    r = roles[dataset_name]
    all_predictors = r["categorical"] + r["ordinal"] + r["numerical"]

    if selected_features is None:
        selected_features = all_predictors

    selected_features = list(selected_features)

    selected_cat = [f for f in r["categorical"] if f in selected_features]
    selected_ord = [f for f in r["ordinal"] if f in selected_features]
    selected_num = [f for f in r["numerical"] if f in selected_features]

    transformers = []

    if selected_num:
        if mode == "chi2":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        elif mode == "tree":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ])
        else:
            raise ValueError(mode)

        transformers.append(("num", num_pipe, selected_num))

    if selected_ord:
        if mode == "chi2":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        elif mode == "tree":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
            ])
        else:
            raise ValueError(mode)

        transformers.append(("ord", ord_pipe, selected_ord))

    if selected_cat:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ])
        transformers.append(("cat", cat_pipe, selected_cat))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True,
    )


## 3. Dynamic transformed-to-source mapping and frozen Chi-Square ranking

In [5]:

def feature_map_from_fitted_preprocessor(
    fitted_preprocessor,
    dataset_name,
    selected_features=None,
):
    r = roles[dataset_name]
    all_predictors = r["categorical"] + r["ordinal"] + r["numerical"]

    if selected_features is None:
        selected_features = all_predictors

    selected_features = list(selected_features)
    selected_num = [f for f in r["numerical"] if f in selected_features]
    selected_ord = [f for f in r["ordinal"] if f in selected_features]
    selected_cat = [f for f in r["categorical"] if f in selected_features]

    rows = []
    idx = 0

    for feature in selected_num:
        rows.append({
            "transformed_index": idx,
            "source_feature": feature,
            "source_role": "numerical",
        })
        idx += 1

    for feature in selected_ord:
        rows.append({
            "transformed_index": idx,
            "source_feature": feature,
            "source_role": "ordinal",
        })
        idx += 1

    if selected_cat:
        cat_pipeline = fitted_preprocessor.named_transformers_["cat"]
        encoder = cat_pipeline.named_steps["onehot"]

        for source_feature, categories in zip(
            selected_cat,
            encoder.categories_,
        ):
            for _ in categories:
                rows.append({
                    "transformed_index": idx,
                    "source_feature": source_feature,
                    "source_role": "categorical",
                })
                idx += 1

    fmap = pd.DataFrame(rows)

    assert len(fmap) == len(
        fitted_preprocessor.get_feature_names_out()
    )

    return fmap


def group_aware_chi2_top75(dataset_name, X_train, y_train):
    prep = build_preprocessor(dataset_name, "chi2")

    X_chi = prep.fit_transform(X_train, y_train)
    assert np.asarray(X_chi).min() >= -1e-12

    fmap = feature_map_from_fitted_preprocessor(
        prep,
        dataset_name,
    )

    raw_scores, _ = chi2(X_chi, y_train)

    temp = fmap.copy()
    temp["raw_score"] = (
        pd.Series(raw_scores)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy()
    )

    n = len(temp)
    transformed_rank = temp["raw_score"].rank(
        ascending=False,
        method="average",
    )

    if n > 1:
        temp["normalized_relevance"] = (
            1.0 - (transformed_rank - 1.0) / (n - 1.0)
        )
    else:
        temp["normalized_relevance"] = 1.0

    grouped = (
        temp.groupby("source_feature", as_index=False)
        .agg(group_score=("normalized_relevance", "mean"))
    )

    grouped["source_rank"] = grouped["group_score"].rank(
        ascending=False,
        method="average",
    )

    grouped = grouped.sort_values(
        ["source_rank", "source_feature"]
    ).reset_index(drop=True)

    p = len(grouped)
    n_select = int(math.ceil(p * FROZEN_FRACTION))

    return grouped["source_feature"].astype(str).tolist()[:n_select]


## 4. Recreate and verify the exact Step-3 outer folds

In [6]:

def build_verified_splits(dataset_name, df):
    r = roles[dataset_name]
    predictors = r["categorical"] + r["ordinal"] + r["numerical"]

    X = df[predictors].copy()
    y = df["adverse_target"].astype(int).copy()
    groups = df["profile_group_id"].astype(str).copy()

    saved = baseline_predictions[
        (baseline_predictions["dataset"] == dataset_name)
        & (baseline_predictions["model"] == "XGB")
    ].copy()

    split_dict = {}

    for repeat_no, seed in enumerate(REPEAT_SEEDS, start=1):
        splitter = StratifiedGroupKFold(
            n_splits=N_FOLDS,
            shuffle=True,
            random_state=seed,
        )

        for fold_no, (train_idx, test_idx) in enumerate(
            splitter.split(X, y, groups),
            start=1,
        ):
            run_id = f"R{repeat_no}_F{fold_no}"

            expected_test = set(np.asarray(test_idx, dtype=int).tolist())
            saved_test = set(
                saved.loc[
                    saved["run_id"] == run_id,
                    "source_row_index",
                ].astype(int).tolist()
            )

            assert expected_test == saved_test
            assert len(
                set(groups.iloc[train_idx]).intersection(
                    set(groups.iloc[test_idx])
                )
            ) == 0

            split_dict[run_id] = {
                "repeat": repeat_no,
                "fold": fold_no,
                "seed": seed,
                "train_idx": np.asarray(train_idx, dtype=int),
                "test_idx": np.asarray(test_idx, dtype=int),
            }

    return split_dict


splits = {
    name: build_verified_splits(name, df)
    for name, df in datasets.items()
}

print("All Step-6 outer folds exactly match Step 3.")


All Step-6 outer folds exactly match Step 3.


## 5. Metrics, expected-cost sensitivity, and fixed XGBoost

In [7]:

def fresh_xgb(scale_pos_weight):
    return XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=float(scale_pos_weight),
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )


def calculate_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    gmean = (
        math.sqrt(specificity * sensitivity)
        if not np.isnan(specificity + sensitivity)
        else np.nan
    )

    fpr, tpr, _ = roc_curve(y_true, y_score)
    ks = float(np.max(tpr - fpr))

    n = len(y_true)

    # Cost per 100 applicants.
    costs = {}
    for fn_cost in [1, 2, 5, 10]:
        total_cost = fp + fn_cost * fn
        costs[f"cost_FN{fn_cost}_FP1_per100"] = 100.0 * total_cost / n

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_adverse": precision_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "recall_adverse": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "specificity": specificity,
        "f1_adverse": f1_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
        "gmean": gmean,
        "ks_statistic": ks,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        **costs,
    }


def training_scale_pos_weight(y_train, strategy_name):
    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())

    if strategy_name == "Unweighted":
        return 1.0

    base_ratio = n_neg / n_pos
    multiplier = WEIGHT_STRATEGIES[strategy_name]

    return base_ratio * multiplier


## 6. Run the 2 × 4 ablation

Per dataset:

- 2 feature regimes
- 4 training-cost regimes
- 25 outer runs

Total: **600 XGBoost evaluations** across the three datasets.

The frozen Chi-Square ranking is fitted once per outer-training fold and reused for its four weighting strategies.


In [8]:

result_rows = []
selected_set_rows = []

for dataset_name, df in datasets.items():
    r = roles[dataset_name]
    all_features = r["categorical"] + r["ordinal"] + r["numerical"]

    X = df[all_features].copy()
    y = df["adverse_target"].astype(int).copy()

    print("\n" + "=" * 80)
    print(dataset_name)
    print("=" * 80)

    for run_number, (run_id, info) in enumerate(
        splits[dataset_name].items(),
        start=1,
    ):
        train_idx = info["train_idx"]
        test_idx = info["test_idx"]

        X_train_all = X.iloc[train_idx].copy()
        X_test_all = X.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        fs_start = time.perf_counter()
        frozen_selected = group_aware_chi2_top75(
            dataset_name,
            X_train_all,
            y_train,
        )
        fs_runtime = time.perf_counter() - fs_start

        selected_set_rows.append({
            "dataset": dataset_name,
            "run_id": run_id,
            "repeat": info["repeat"],
            "fold": info["fold"],
            "selected_source_features": len(frozen_selected),
            "selected_features": ";".join(frozen_selected),
            "feature_selection_runtime_seconds": fs_runtime,
        })

        feature_regime_map = {
            "AllFeatures": all_features,
            "FrozenChi2Top75": frozen_selected,
        }

        print(f"\n{run_id} ({run_number}/25)")

        for feature_regime, selected_features in feature_regime_map.items():
            for weight_strategy in WEIGHT_STRATEGIES:
                spw = training_scale_pos_weight(
                    y_train,
                    weight_strategy,
                )

                prep = build_preprocessor(
                    dataset_name,
                    "tree",
                    selected_features=selected_features,
                )

                pipe = Pipeline([
                    ("preprocessor", prep),
                    ("model", fresh_xgb(spw)),
                ])

                start = time.perf_counter()

                pipe.fit(
                    X_train_all[selected_features],
                    y_train,
                )

                y_pred = pipe.predict(
                    X_test_all[selected_features]
                )

                y_score = pipe.predict_proba(
                    X_test_all[selected_features]
                )[:, 1]

                runtime = time.perf_counter() - start

                metric_values = calculate_metrics(
                    y_test,
                    y_pred,
                    y_score,
                )

                transformed_count = len(
                    pipe.named_steps[
                        "preprocessor"
                    ].get_feature_names_out()
                )

                result_rows.append({
                    "dataset": dataset_name,
                    "run_id": run_id,
                    "repeat": info["repeat"],
                    "fold": info["fold"],
                    "feature_regime": feature_regime,
                    "weight_strategy": weight_strategy,
                    "scale_pos_weight": spw,
                    "source_features": len(selected_features),
                    "transformed_features": transformed_count,
                    "feature_selection_runtime_seconds": (
                        fs_runtime
                        if feature_regime == "FrozenChi2Top75"
                        else 0.0
                    ),
                    "model_runtime_seconds": runtime,
                    **metric_values,
                })

                print(
                    f"{feature_regime:16s} | "
                    f"{weight_strategy:10s} | "
                    f"SPW={spw:.3f} | "
                    f"Recall={metric_values['recall_adverse']:.3f} | "
                    f"F1={metric_values['f1_adverse']:.3f} | "
                    f"MCC={metric_values['mcc']:.3f}"
                )


results = pd.DataFrame(result_rows)
selected_sets = pd.DataFrame(selected_set_rows)

results.to_csv(
    OUT_DIR / "cost_sensitive_xgb_fold_results.csv",
    index=False,
)

selected_sets.to_csv(
    OUT_DIR / "cost_sensitive_selected_feature_sets.csv",
    index=False,
)

print("\nStep-6 model runs completed.")



Australian Credit Approval

R1_F1 (1/25)
AllFeatures      | Unweighted | SPW=1.000 | Recall=0.815 | F1=0.857 | MCC=0.683
AllFeatures      | Balanced   | SPW=0.828 | Recall=0.815 | F1=0.863 | MCC=0.699
AllFeatures      | Cost2      | SPW=1.656 | Recall=0.815 | F1=0.841 | MCC=0.633
AllFeatures      | Cost5      | SPW=4.139 | Recall=0.864 | F1=0.864 | MCC=0.671
FrozenChi2Top75  | Unweighted | SPW=1.000 | Recall=0.827 | F1=0.870 | MCC=0.712
FrozenChi2Top75  | Balanced   | SPW=0.828 | Recall=0.827 | F1=0.870 | MCC=0.712
FrozenChi2Top75  | Cost2      | SPW=1.656 | Recall=0.827 | F1=0.870 | MCC=0.712
FrozenChi2Top75  | Cost5      | SPW=4.139 | Recall=0.840 | F1=0.850 | MCC=0.643

R1_F2 (2/25)
AllFeatures      | Unweighted | SPW=1.000 | Recall=0.840 | F1=0.872 | MCC=0.708
AllFeatures      | Balanced   | SPW=0.828 | Recall=0.840 | F1=0.877 | MCC=0.725
AllFeatures      | Cost2      | SPW=1.656 | Recall=0.852 | F1=0.857 | MCC=0.657
AllFeatures      | Cost5      | SPW=4.139 | Recall=0.889 | F1=0.

## 7. Reproduction checks against Steps 3–5

In [9]:

# Check AllFeatures + Unweighted against Step-3 XGB.
step3_xgb = baseline_results[
    baseline_results["model"] == "XGB"
][
    ["dataset", "run_id", "roc_auc", "pr_auc", "mcc"]
].copy()

current_all = results[
    (results["feature_regime"] == "AllFeatures")
    & (results["weight_strategy"] == "Unweighted")
][
    ["dataset", "run_id", "roc_auc", "pr_auc", "mcc"]
].copy()

check_all = current_all.merge(
    step3_xgb,
    on=["dataset", "run_id"],
    suffixes=("_step6", "_step3"),
    validate="one_to_one",
)

for metric in ["roc_auc", "pr_auc", "mcc"]:
    max_abs = np.max(
        np.abs(
            check_all[f"{metric}_step6"]
            - check_all[f"{metric}_step3"]
        )
    )
    print("AllFeatures reproduction", metric, "max abs diff =", max_abs)
    assert max_abs < 1e-10


# Check FrozenFS + Unweighted against Step 4/5.
reference_fs = pd.concat([
    step4_results[
        (step4_results["selector"] == "Chi2")
        & (step4_results["subset"] == "Top75")
    ].assign(dataset="German Credit"),
    step5_results,
], ignore_index=True)

reference_fs = reference_fs[
    ["dataset", "run_id", "roc_auc", "pr_auc", "mcc"]
].copy()

current_fs = results[
    (results["feature_regime"] == "FrozenChi2Top75")
    & (results["weight_strategy"] == "Unweighted")
][
    ["dataset", "run_id", "roc_auc", "pr_auc", "mcc"]
].copy()

check_fs = current_fs.merge(
    reference_fs,
    on=["dataset", "run_id"],
    suffixes=("_step6", "_reference"),
    validate="one_to_one",
)

for metric in ["roc_auc", "pr_auc", "mcc"]:
    max_abs = np.max(
        np.abs(
            check_fs[f"{metric}_step6"]
            - check_fs[f"{metric}_reference"]
        )
    )
    print("FrozenFS reproduction", metric, "max abs diff =", max_abs)
    assert max_abs < 1e-10

print("\nAll Step-6 reproduction checks passed.")


AllFeatures reproduction roc_auc max abs diff = 1.1102230246251565e-16
AllFeatures reproduction pr_auc max abs diff = 1.1102230246251565e-16
AllFeatures reproduction mcc max abs diff = 5.551115123125783e-17
FrozenFS reproduction roc_auc max abs diff = 1.1102230246251565e-16
FrozenFS reproduction pr_auc max abs diff = 1.1102230246251565e-16
FrozenFS reproduction mcc max abs diff = 5.551115123125783e-17

All Step-6 reproduction checks passed.


## 8. Summary table

In [10]:

summary = (
    results
    .groupby(
        [
            "dataset",
            "feature_regime",
            "weight_strategy",
        ],
        as_index=False,
    )
    .agg(
        Scale_Pos_Weight_Mean=("scale_pos_weight", "mean"),
        Source_Features=("source_features", "mean"),
        ROC_AUC=("roc_auc", "mean"),
        PR_AUC=("pr_auc", "mean"),
        Recall=("recall_adverse", "mean"),
        Precision=("precision_adverse", "mean"),
        F1=("f1_adverse", "mean"),
        Balanced_Accuracy=("balanced_accuracy", "mean"),
        MCC=("mcc", "mean"),
        GMean=("gmean", "mean"),
        KS=("ks_statistic", "mean"),
        Cost_1_1=("cost_FN1_FP1_per100", "mean"),
        Cost_2_1=("cost_FN2_FP1_per100", "mean"),
        Cost_5_1=("cost_FN5_FP1_per100", "mean"),
        Cost_10_1=("cost_FN10_FP1_per100", "mean"),
        Runtime_Seconds=("model_runtime_seconds", "mean"),
    )
)

summary.to_csv(
    OUT_DIR / "cost_sensitive_xgb_summary.csv",
    index=False,
)

for dataset_name in datasets:
    print("\n", dataset_name)
    display(
        summary[
            summary["dataset"] == dataset_name
        ].sort_values(
            ["MCC", "ROC_AUC"],
            ascending=False,
        )
    )



 Australian Credit Approval


,dataset,feature_regime,weight_strategy,Scale_Pos_Weight_Mean,Source_Features,ROC_AUC,PR_AUC,Recall,Precision,F1,Balanced_Accuracy,MCC,GMean,KS,Cost_1_1,Cost_2_1,Cost_5_1,Cost_10_1,Runtime_Seconds
0,Australian Credit Approval,AllFeatures,Balanced,0.802081,14.0,0.932256,0.933074,0.877836,0.893418,0.884846,0.874390,0.746420,0.873996,0.777686,12.608696,19.420290,39.855072,73.913043,0.270693
3,Australian Credit Approval,AllFeatures,Unweighted,1.000000,14.0,0.931568,0.932837,0.878958,0.889353,0.883336,0.872130,0.742331,0.871680,0.777197,12.811594,19.565217,39.826087,73.594203,0.274358
1,Australian Credit Approval,AllFeatures,Cost2,1.604162,14.0,0.931033,0.933169,0.894584,0.872472,0.882670,0.866375,0.734636,0.865511,0.776278,13.130435,19.014493,36.666667,66.086957,0.275720
2,Australian Credit Approval,AllFeatures,Cost5,4.010405,14.0,0.930363,0.932258,0.920275,0.850410,0.883263,0.859052,0.727994,0.856378,0.774593,13.478261,17.942029,31.333333,53.652174,0.278581
4,Australian Credit Approval,FrozenChi2Top75,Balanced,0.802081,11.0,0.926164,0.929792,0.861221,0.884285,0.871964,0.860281,0.718390,0.859874,0.762315,13.971014,21.681159,44.811594,83.362319,0.249773
7,Australian Credit Approval,FrozenChi2Top75,Unweighted,1.000000,11.0,0.925940,0.930499,0.871046,0.876971,0.873366,0.859510,0.717740,0.859070,0.760597,13.971014,21.159420,42.724638,78.666667,0.252971
5,Australian Credit Approval,FrozenChi2Top75,Cost2,1.604162,11.0,0.925344,0.930712,0.883962,0.864670,0.873341,0.855385,0.713129,0.854284,0.760753,14.173913,20.637681,40.028986,72.347826,0.256136
6,Australian Credit Approval,FrozenChi2Top75,Cost5,4.010405,11.0,0.924304,0.929814,0.907816,0.838190,0.870586,0.844614,0.699287,0.841507,0.752605,14.927536,20.086957,35.565217,61.362319,0.258545



 German Credit


,dataset,feature_regime,weight_strategy,Scale_Pos_Weight_Mean,Source_Features,ROC_AUC,PR_AUC,Recall,Precision,F1,Balanced_Accuracy,MCC,GMean,KS,Cost_1_1,Cost_2_1,Cost_5_1,Cost_10_1,Runtime_Seconds
12,German Credit,FrozenChi2Top75,Balanced,2.335805,15.0,0.787275,0.619312,0.641251,0.565414,0.598078,0.714983,0.415944,0.709594,0.485243,25.56,36.32,68.60,122.40,0.272916
9,German Credit,AllFeatures,Cost2,4.671609,20.0,0.786407,0.627760,0.706191,0.532656,0.605155,0.720148,0.412008,0.719224,0.474902,27.46,36.30,62.82,107.02,0.291917
8,German Credit,AllFeatures,Balanced,2.335805,20.0,0.790088,0.632007,0.608736,0.569841,0.586398,0.705714,0.404344,0.697714,0.477794,25.50,37.18,72.22,130.62,0.287680
13,German Credit,FrozenChi2Top75,Cost2,4.671609,15.0,0.783545,0.614073,0.718758,0.519751,0.600787,0.717211,0.403949,0.715940,0.476092,28.38,36.84,62.22,104.52,0.272634
11,German Credit,AllFeatures,Unweighted,1.000000,20.0,0.790369,0.632366,0.478338,0.640146,0.544334,0.681676,0.398821,0.648509,0.478771,23.68,39.30,86.16,164.26,0.274282
15,German Credit,FrozenChi2Top75,Unweighted,1.000000,15.0,0.788434,0.622283,0.490576,0.632947,0.547579,0.683953,0.398727,0.653594,0.485773,23.92,39.22,85.12,161.62,0.270055
10,German Credit,AllFeatures,Cost5,11.679023,20.0,0.784000,0.623481,0.768241,0.482679,0.591089,0.707756,0.380828,0.704212,0.471806,31.66,38.64,59.58,94.48,0.286374
14,German Credit,FrozenChi2Top75,Cost5,11.679023,15.0,0.781219,0.614738,0.783872,0.474679,0.589416,0.705838,0.377213,0.700558,0.470182,32.56,39.06,58.56,91.06,0.273728



 Taiwan Credit Card Default


,dataset,feature_regime,weight_strategy,Scale_Pos_Weight_Mean,Source_Features,ROC_AUC,PR_AUC,Recall,Precision,F1,Balanced_Accuracy,MCC,GMean,KS,Cost_1_1,Cost_2_1,Cost_5_1,Cost_10_1,Runtime_Seconds
23,Taiwan Credit Card Default,FrozenChi2Top75,Unweighted,1.000000,18.0,0.782617,0.560105,0.366999,0.674938,0.475385,0.658392,0.404185,0.590326,0.436166,17.914005,31.917332,73.927314,143.943949,0.862918
19,Taiwan Credit Card Default,AllFeatures,Unweighted,1.000000,23.0,0.783234,0.560525,0.367964,0.671547,0.475333,0.658417,0.402911,0.590805,0.436558,17.964002,31.946002,73.892000,143.801996,0.936144
16,Taiwan Credit Card Default,AllFeatures,Balanced,3.520846,23.0,0.782980,0.560095,0.628899,0.471281,0.538707,0.714210,0.389869,0.709041,0.434786,23.822602,32.031977,56.660102,97.706977,0.973255
20,Taiwan Credit Card Default,FrozenChi2Top75,Balanced,3.520846,18.0,0.782551,0.559242,0.629868,0.470737,0.538683,0.714306,0.389748,0.709232,0.435487,23.861918,32.049967,56.614116,97.554363,0.858638
17,Taiwan Credit Card Default,AllFeatures,Cost2,7.041693,23.0,0.782127,0.559147,0.823895,0.334588,0.475871,0.679239,0.298781,0.663607,0.433127,40.143269,44.039962,55.730038,75.213499,0.984821
21,Taiwan Credit Card Default,FrozenChi2Top75,Cost2,7.041693,18.0,0.781627,0.558023,0.828581,0.331336,0.473343,0.676827,0.295244,0.659548,0.433733,40.780654,44.574035,55.954176,74.921078,0.859852
18,Taiwan Credit Card Default,AllFeatures,Cost5,17.604232,23.0,0.779882,0.556909,0.953189,0.254577,0.401807,0.580210,0.176576,0.444306,0.432184,62.778711,63.815391,66.925430,72.108828,0.989194
22,Taiwan Credit Card Default,FrozenChi2Top75,Cost5,17.604232,18.0,0.779959,0.556771,0.956311,0.253814,0.401132,0.578865,0.175750,0.438761,0.431299,63.162056,64.129403,67.031443,71.868176,0.844127


## 9. German-only development decision aid

To avoid selecting a training-cost rule from external test results, this table isolates **German Credit**.

The final weighting strategy should be chosen only after reviewing German:

- adverse recall,
- precision and F1,
- MCC,
- balanced accuracy,
- discrimination,
- assumed-cost sensitivity.

Australian and Taiwan results remain replication evidence, not the basis for choosing the weight.


In [11]:

german_decision = summary[
    summary["dataset"] == "German Credit"
].copy()

german_decision.to_csv(
    OUT_DIR / "german_cost_sensitive_decision_table.csv",
    index=False,
)

display(
    german_decision.sort_values(
        ["MCC", "Balanced_Accuracy", "ROC_AUC"],
        ascending=False,
    )
)


,dataset,feature_regime,weight_strategy,Scale_Pos_Weight_Mean,Source_Features,ROC_AUC,PR_AUC,Recall,Precision,F1,Balanced_Accuracy,MCC,GMean,KS,Cost_1_1,Cost_2_1,Cost_5_1,Cost_10_1,Runtime_Seconds
12,German Credit,FrozenChi2Top75,Balanced,2.335805,15.0,0.787275,0.619312,0.641251,0.565414,0.598078,0.714983,0.415944,0.709594,0.485243,25.56,36.32,68.60,122.40,0.272916
9,German Credit,AllFeatures,Cost2,4.671609,20.0,0.786407,0.627760,0.706191,0.532656,0.605155,0.720148,0.412008,0.719224,0.474902,27.46,36.30,62.82,107.02,0.291917
8,German Credit,AllFeatures,Balanced,2.335805,20.0,0.790088,0.632007,0.608736,0.569841,0.586398,0.705714,0.404344,0.697714,0.477794,25.50,37.18,72.22,130.62,0.287680
13,German Credit,FrozenChi2Top75,Cost2,4.671609,15.0,0.783545,0.614073,0.718758,0.519751,0.600787,0.717211,0.403949,0.715940,0.476092,28.38,36.84,62.22,104.52,0.272634
11,German Credit,AllFeatures,Unweighted,1.000000,20.0,0.790369,0.632366,0.478338,0.640146,0.544334,0.681676,0.398821,0.648509,0.478771,23.68,39.30,86.16,164.26,0.274282
15,German Credit,FrozenChi2Top75,Unweighted,1.000000,15.0,0.788434,0.622283,0.490576,0.632947,0.547579,0.683953,0.398727,0.653594,0.485773,23.92,39.22,85.12,161.62,0.270055
10,German Credit,AllFeatures,Cost5,11.679023,20.0,0.784000,0.623481,0.768241,0.482679,0.591089,0.707756,0.380828,0.704212,0.471806,31.66,38.64,59.58,94.48,0.286374
14,German Credit,FrozenChi2Top75,Cost5,11.679023,15.0,0.781219,0.614738,0.783872,0.474679,0.589416,0.705838,0.377213,0.700558,0.470182,32.56,39.06,58.56,91.06,0.273728


## 10. Weight–feature interaction deltas

In [12]:

reference = results[
    (results["feature_regime"] == "AllFeatures")
    & (results["weight_strategy"] == "Unweighted")
][
    [
        "dataset",
        "run_id",
        "roc_auc",
        "pr_auc",
        "recall_adverse",
        "precision_adverse",
        "f1_adverse",
        "balanced_accuracy",
        "mcc",
        "gmean",
        "ks_statistic",
        "cost_FN1_FP1_per100",
        "cost_FN2_FP1_per100",
        "cost_FN5_FP1_per100",
        "cost_FN10_FP1_per100",
    ]
].copy()

reference = reference.rename(
    columns={
        c: "reference_" + c
        for c in reference.columns
        if c not in ["dataset", "run_id"]
    }
)

paired = results.merge(
    reference,
    on=["dataset", "run_id"],
    how="left",
    validate="many_to_one",
)

metrics_for_delta = [
    "roc_auc",
    "pr_auc",
    "recall_adverse",
    "precision_adverse",
    "f1_adverse",
    "balanced_accuracy",
    "mcc",
    "gmean",
    "ks_statistic",
    "cost_FN1_FP1_per100",
    "cost_FN2_FP1_per100",
    "cost_FN5_FP1_per100",
    "cost_FN10_FP1_per100",
]

for metric in metrics_for_delta:
    paired["delta_" + metric] = (
        paired[metric]
        - paired["reference_" + metric]
    )

paired.to_csv(
    OUT_DIR / "cost_sensitive_xgb_paired_deltas.csv",
    index=False,
)

delta_summary = (
    paired.groupby(
        [
            "dataset",
            "feature_regime",
            "weight_strategy",
        ],
        as_index=False,
    )
    .agg(
        Delta_ROC_AUC=("delta_roc_auc", "mean"),
        Delta_PR_AUC=("delta_pr_auc", "mean"),
        Delta_Recall=("delta_recall_adverse", "mean"),
        Delta_Precision=("delta_precision_adverse", "mean"),
        Delta_F1=("delta_f1_adverse", "mean"),
        Delta_Balanced_Accuracy=("delta_balanced_accuracy", "mean"),
        Delta_MCC=("delta_mcc", "mean"),
        Delta_Cost_2_1=("delta_cost_FN2_FP1_per100", "mean"),
        Delta_Cost_5_1=("delta_cost_FN5_FP1_per100", "mean"),
        Delta_Cost_10_1=("delta_cost_FN10_FP1_per100", "mean"),
    )
)

delta_summary.to_csv(
    OUT_DIR / "cost_sensitive_xgb_delta_summary.csv",
    index=False,
)

display(delta_summary)


,dataset,feature_regime,weight_strategy,Delta_ROC_AUC,Delta_PR_AUC,Delta_Recall,Delta_Precision,Delta_F1,Delta_Balanced_Accuracy,Delta_MCC,Delta_Cost_2_1,Delta_Cost_5_1,Delta_Cost_10_1
0,Australian Credit Approval,AllFeatures,Balanced,0.000689,0.000238,-0.001121,0.004065,0.001510,0.002260,0.004089,-0.144928,0.028986,0.318841
1,Australian Credit Approval,AllFeatures,Cost2,-0.000535,0.000333,0.015627,-0.016881,-0.000666,-0.005755,-0.007695,-0.550725,-3.159420,-7.507246
2,Australian Credit Approval,AllFeatures,Cost5,-0.001204,-0.000579,0.041317,-0.038943,-0.000073,-0.013078,-0.014338,-1.623188,-8.492754,-19.942029
3,Australian Credit Approval,AllFeatures,Unweighted,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,Australian Credit Approval,FrozenChi2Top75,Balanced,-0.005403,-0.003045,-0.017737,-0.005069,-0.011372,-0.011849,-0.023942,2.115942,4.985507,9.768116
5,Australian Credit Approval,FrozenChi2Top75,Cost2,-0.006223,-0.002125,0.005004,-0.024683,-0.009995,-0.016745,-0.029203,1.072464,0.202899,-1.246377
6,Australian Credit Approval,FrozenChi2Top75,Cost5,-0.007264,-0.003022,0.028859,-0.051163,-0.012750,-0.027516,-0.043045,0.521739,-4.260870,-12.231884
7,Australian Credit Approval,FrozenChi2Top75,Unweighted,-0.005628,-0.002338,-0.007912,-0.012382,-0.009970,-0.012620,-0.024592,1.594203,2.898551,5.072464
8,German Credit,AllFeatures,Balanced,-0.000281,-0.000359,0.130398,-0.070304,0.042064,0.024038,0.005523,-2.120000,-13.940000,-33.640000
9,German Credit,AllFeatures,Cost2,-0.003962,-0.004606,0.227853,-0.107490,0.060821,0.038472,0.013187,-3.000000,-23.340000,-57.240000


## 11. Final consistency checks

In [13]:

expected_rows = 3 * 2 * 4 * 25

assert len(results) == expected_rows, (
    f"Expected {expected_rows} rows; found {len(results)}."
)

assert results["roc_auc"].between(0, 1).all()
assert results["pr_auc"].between(0, 1).all()
assert results["mcc"].between(-1, 1).all()

assert not results[
    [
        "roc_auc",
        "pr_auc",
        "recall_adverse",
        "f1_adverse",
        "balanced_accuracy",
        "mcc",
    ]
].isna().any().any()

configuration = {
    "stage": "Objective 3 Step 6 - cost-sensitive XGBoost ablation",
    "feature_regimes": FEATURE_REGIMES,
    "training_weight_strategies": {
        "Unweighted": "scale_pos_weight = 1",
        "Balanced": "N_negative / N_positive",
        "Cost2": "2 * N_negative / N_positive",
        "Cost5": "5 * N_negative / N_positive",
    },
    "weight_estimation": "outer-training fold only",
    "feature_selection": (
        "Frozen group-aware Chi-Square Top75, fitted only on outer-training fold"
    ),
    "comparison_feature_regime": "All Features",
    "threshold": 0.5,
    "calibration": "none",
    "hyperparameter_tuning": "none",
    "decision_rule": (
        "Training-cost strategy will be chosen using German evidence only; "
        "Australian and Taiwan are replication evidence."
    ),
}

with open(
    OUT_DIR / "step6_experiment_configuration.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(configuration, f, indent=4)

manifest = sorted(
    [p.name for p in OUT_DIR.iterdir() if p.is_file()]
)

pd.DataFrame(
    {"generated_file": manifest}
).to_csv(
    OUT_DIR / "step6_output_manifest.csv",
    index=False,
)

print("=" * 80)
print("STEP 6 COMPLETED SUCCESSFULLY")
print("=" * 80)
print("Output folder:", OUT_DIR)
print("\nUpload the complete folder after execution.")


STEP 6 COMPLETED SUCCESSFULLY
Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\cost_sensitive_xgb_ablation

Upload the complete folder after execution.
